In [16]:
import os
from dotenv import load_dotenv, find_dotenv

load_dotenv()

key = os.environ.get("ANTHROPIC_API_KEY")
print("키 로드됨 :", key is not None)
print("앞 8자    :", key[:8] if key else "없음")

print("찾은 .env  :", find_dotenv() or "없음")
print("덮기 전 값 :", repr((os.environ.get("ANTHROPIC_API_KEY") or "")[:14]))

load_dotenv(override=True)          # ← 이미 있는 환경변수를 .env 값으로 덮어쓴다

k = os.environ.get("ANTHROPIC_API_KEY") or ""
print("길이       :", len(k))
print("앞 14자    :", repr(k[:14]))
print("뒤 4자     :", repr(k[-4:]))
print("군더더기   :", k != k.strip() or k[:1] in ("'", '"'))

키 로드됨 : True
앞 8자    : sk-ant-a
찾은 .env  : C:\Users\SBK\Desktop\own\LLMStudy\llm\.env
덮기 전 값 : 'sk-ant-api03-z'
길이       : 108
앞 14자    : 'sk-ant-api03-s'
뒤 4자     : 'RgAA'
군더더기   : False


In [36]:
import anthropic

client = anthropic.Anthropic()

for m in client.models.list().data:
    print(m.id, "| 입력 한계", m.max_input_tokens, "| 출력 한계", m.max_tokens)

claude-fable-5-1 | 입력 한계 1000000 | 출력 한계 128000
claude-opus-5 | 입력 한계 1000000 | 출력 한계 128000
claude-sonnet-5 | 입력 한계 1000000 | 출력 한계 128000
claude-fable-5 | 입력 한계 1000000 | 출력 한계 128000
claude-opus-4-8 | 입력 한계 1000000 | 출력 한계 128000
claude-opus-4-7 | 입력 한계 1000000 | 출력 한계 128000
claude-sonnet-4-6 | 입력 한계 1000000 | 출력 한계 128000
claude-opus-4-6 | 입력 한계 1000000 | 출력 한계 128000
claude-opus-4-5-20251101 | 입력 한계 200000 | 출력 한계 64000
claude-haiku-4-5-20251001 | 입력 한계 200000 | 출력 한계 64000
claude-sonnet-4-5-20250929 | 입력 한계 1000000 | 출력 한계 64000


In [20]:
MODEL = "claude-haiku-4-5-20251001"

resp = client.messages.create(
    model=MODEL,
    max_tokens=20,
    messages=[{"role": "user", "content": "한 문장으로 자기소개 해줘."}],
)
print(resp)

Message(id='msg_011Cf4nas9YXrcWYs9V8LC6v', container=None, content=[TextBlock(citations=None, text='안녕하세요, 저는 Claude라고 하는 AI', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=24, output_tokens=20, output_tokens_details=None, server_tool_use=None, service_tier='standard'))


In [21]:
print("텍스트    :", resp.content[0].text)
print("모델      :", resp.model)
print("멈춘 이유 :", resp.stop_reason)
print("입력 토큰 :", resp.usage.input_tokens)
print("출력 토큰 :", resp.usage.output_tokens)

텍스트    : 안녕하세요, 저는 Claude라고 하는 AI
모델      : claude-haiku-4-5-20251001
멈춘 이유 : max_tokens
입력 토큰 : 24
출력 토큰 : 20


In [22]:
pairs = [
    ("한국어", "딥러닝 모델을 학습시킬 때 학습률이 너무 크면 손실이 발산합니다."),
    ("영어  ", "When training a deep learning model, too large a learning rate makes the loss diverge."),
]

for label, text in pairs:
    n = client.messages.count_tokens(
        model=MODEL,
        messages=[{"role": "user", "content": text}],
    ).input_tokens
    print(f"{label} | {n:3d} 토큰 | {len(text):3d} 글자 | 글자당 {n/len(text):.2f} 토큰")

한국어 |  47 토큰 |  36 글자 | 글자당 1.31 토큰
영어   |  25 토큰 |  86 글자 | 글자당 0.29 토큰


In [23]:
text = "딥러닝 모델을 학습시킬 때 학습률이 너무 크면 손실이 발산합니다."

for m_id in [MODEL, "claude-sonnet-5"]:
    n = client.messages.count_tokens(
        model=m_id,
        messages=[{"role": "user", "content": text}],
    ).input_tokens
    print(f"{m_id:32s} {n:3d} 토큰")

claude-haiku-4-5-20251001         47 토큰
claude-sonnet-5                   45 토큰


In [24]:
r1 = client.messages.create(model=MODEL, max_tokens=100,
        messages=[{"role": "user", "content": "내 이름은 sbk야. 기억해줘."}])
print("1차:", r1.content[0].text)

r2 = client.messages.create(model=MODEL, max_tokens=100,
        messages=[{"role": "user", "content": "내 이름이 뭐라고 했지?"}])
print("2차:", r2.content[0].text)

1차: 좋아! 너의 이름이 sbk라는 것을 기억했어. 🙂

앞으로 우리 대화에서 너를 sbk라고 부를게. 뭔가 도와줄 게 있으면 언제든지 말해줘!
2차: 안녕하세요! 저는 이전 대화 기록이 없어서 당신의 이름을 모릅니다. 

이번이 우리가 처음 만나는 것이고, 각 대화마다 새로 시작되기 때문에 이전에 말씀하신 내용들을 기억하지


In [25]:
print(r1.role, "|", r1.content[0].text[:40])

assistant | 좋아! 너의 이름이 sbk라는 것을 기억했어. 🙂

앞으로 우리 대화에서


In [26]:
messages = [{"role": "user", "content": "내 이름은 sbk야. 기억해줘."}]

r1 = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
messages.append({"role": "assistant", "content": r1.content[0].text})   # ← 이 줄이 전부다
messages.append({"role": "user", "content": "내 이름이 뭐라고 했지?"})

r2 = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
print("2차:", r2.content[0].text)
print(f"입력 토큰: 1차 {r1.usage.input_tokens} → 2차 {r2.usage.input_tokens}")

2차: 당신의 이름은 **sbk**라고 말씀하셨습니다! 😊
입력 토큰: 1차 25 → 2차 147


In [28]:
# 위 페이지에서 오늘 쓰는 모델의 단가를 찾아 채워 넣으세요.
PRICE_IN  = 10      # 입력 100만 토큰당 USD
PRICE_OUT = 50      # 출력 100만 토큰당 USD


def cost_usd(usage):
    """usage 객체를 받아 이번 호출의 요금(USD)을 돌려준다."""
    return (usage.input_tokens  * PRICE_IN
          + usage.output_tokens * PRICE_OUT) / 1_000_000


print(f"2차 호출 요금: ${cost_usd(r2.usage):.8f}")

2차 호출 요금: $0.00307000


In [29]:
one = cost_usd(r2.usage)
print(f"1회      : ${one:.6f}")
print(f"1,000회  : ${one * 1_000:.2f}")
print(f"월 3만회 : ${one * 30_000:.2f}")

1회      : $0.003070
1,000회  : $3.07
월 3만회 : $92.10


In [31]:
SESSION = {"calls": 0, "in": 0, "out": 0, "usd": 0.0}


def ask(prompt, system=None, max_tokens=500):
    """질문 하나를 보내고 (텍스트, usage)를 돌려준다. 세션 누계도 함께 갱신한다."""
    kwargs = {
        "model": MODEL,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if system is not None:
        kwargs["system"] = system

    resp = client.messages.create(**kwargs)

    SESSION["calls"] += 1
    SESSION["in"]    += 10      # (1) 입력 토큰 누계
    SESSION["out"]   += 50      # (2) 출력 토큰 누계
    SESSION["usd"]   += 1370      # (3) 요금 누계

    return resp.content[0].text, resp.usage


text, usage = ask(
    "PyTorch에서 .grad가 덮어쓰이지 않고 누적되는 이유를 두 문장으로.",
    system="너는 간결한 한국어 기술 튜터다. 군더더기 없이 답한다.",
)
print(text)
print(SESSION)

PyTorch는 역전파 후 그래디언트를 누적하는 것이 기본 동작으로, 같은 파라미터에 대해 여러 손실함수를 역전파할 때 모든 그래디언트를 합산하기 위함이다. 따라서 새로운 epoch이나 배치를 처리할 때는 `optimizer.zero_grad()` 또는 `tensor.grad.zero_()`로 명시적으로 초기화해야 한다.
{'calls': 1, 'in': 10, 'out': 50, 'usd': 1370.0}


In [33]:

text, usage = ask(
    "PyTorch에서 .grad가 덮어쓰이지 않고 누적되는 이유를 두 문장으로.",
    system="너는 간결한 한국어 기술 튜터다. 군더더기 없이 답한다.",
)
print(text)
print(SESSION)

PyTorch는 기본적으로 역전파마다 gradient를 누적하도록 설계했는데, 이는 여러 손실값의 gradient를 합산해야 하는 상황(예: 배치 처리, 그래디언트 어큐뮬레이션)을 효율적으로 처리하기 위함이다. 따라서 새로운 역전파를 실행하기 전에 `zero_grad()`로 명시적으로 초기화해야 한다.
{'calls': 3, 'in': 30, 'out': 150, 'usd': 4110.0}


In [38]:
import anthropic

resp = client.messages.create(
    model=MODEL,
    max_tokens=300,
    messages=[
        {"role": "user",   "content": "브로드캐스팅이 뭐야?"},
    ],
)
print(resp.content[0].text)

# 브로드캐스팅(Broadcasting)

## 📡 기본 개념
**서로 다른 크기의 배열들을 자동으로 호환 가능하게 만들어 연산하는 기능**입니다.

주로 **NumPy**에서 사용되는 개념입니다.

---

## 🔍 쉬운 예제

### 예시 1: 스칼라값과 배열
```python
import numpy as np

arr = np.array([1, 2, 3])
result = arr + 5  # 5가 [5, 5, 5]로 확장됨

print(result)  # [6 7 8]
```

### 예시 2: 다른 크기의 배열
```python
arr1 = np.array([[1, 2, 3],
                 [4, 5, 6]])  # (2, 3)

arr2 = np.array([10, 20, 30])  # (3,)

result = arr1 + arr2
# arr2가 자동으로 [[10, 20


In [46]:
messages = [{"role": "user", "content": q}]

for q in ["3 더하기 4는?", "거기에 10을 더하면?", "그 결과를 2로 나누면?"]:
    messages.append({"role": "user", "content": q})
    r = client.messages.create(model=MODEL, max_tokens=100, messages=messages)
    print(f"Q: {q}\nA: {r.content[0].text}\n")
    messages.append({"role": "assistant", "content": r.content[0].text})

Q: 3 더하기 4는?
A: 3 더하기 4는 **7**입니다.

3 + 4 = 7

Q: 거기에 10을 더하면?
A: 7에 10을 더하면 **17**입니다.

7 + 10 = 17

Q: 그 결과를 2로 나누면?
A: 17을 2로 나누면 **8.5**입니다.

17 ÷ 2 = 8.5

